# Zero Lattice — Telperion
## Paper: *How an Addition EQUALS a Subtraction*
### 02 — Data (engine run)

**Date:** 2026-06-25  
**Engine:** `ValaQuenta/zero_lattice.py` v0.100

*Collect all data. Run all engine functions. No interpretation here — that is notebook 03.*

---

In [ ]:
import sys, os
import math
import numpy as np

# Add ValaQuenta to path
sys.path.insert(0, os.path.abspath('../..'))

from zero_lattice import (
    # Constants
    THE_ANGLE, SEDENION_DIM, ZD_CLASSES, ZD_PAIRS,
    OMEGA_ZS, D_STAR, SIGMA_HALF, GAP, SHELL_SIGMA,
    BASIS_MAP, MONSTER_GAP, RIEMANN_ZEROS,
    # Algebra
    build_mul_table, _cd_mul_basis, multiply, e_k, norm_sq,
    # ZD enumeration
    find_zd_pairs, classify_zd_pairs,
    # Sedenion Point Mapping
    basis_to_cell, sedenion_point_map, sphere_coordinates,
    # The Angle
    the_angle, verify_angle, zd_path_coordinates,
    # The Tree
    build_tree, view_tree,
    # Zeta
    zeta_geometric, zeta_dirichlet, critical_line_samples, _zd_weights,
    # Scale switching
    switch_scale,
    # Monster gap
    monster_gap_in_map, laurelin_interface,
    # Universal Translator
    universal_translator_structure,
    # Full run
    run_all,
)

print('Engine loaded. Constants:')
print(f'  THE_ANGLE  = π/8 = {THE_ANGLE:.15f} rad = {math.degrees(THE_ANGLE):.1f}°')
print(f'  ZD_PAIRS   = {ZD_PAIRS}')
print(f'  ZD_CLASSES = {ZD_CLASSES}')
print(f'  OMEGA_ZS   = {OMEGA_ZS}')
print(f'  D_STAR     = {D_STAR}')
print(f'  GAP        = {GAP:.6e}')
print(f'  MONSTER_GAP= {set(MONSTER_GAP)}')

## D1 — Sedenion Multiplication Table

In [ ]:
T_sign, T_idx = build_mul_table()

print('16×16 sedenion multiplication table (sign component):')
print('     ', '  '.join(f'e{j:<2}' for j in range(16)))
for i in range(16):
    row = '  '.join(f'{int(T_sign[i,j]):+2}' for j in range(16))
    print(f'e{i:<2}: {row}')

print()
print('Key checks:')
# e0 is identity
for j in range(16):
    s, k = _cd_mul_basis(0, j, 4)
    assert s == 1 and k == j, f'e0 × e{j} should be e{j}'
print(f'  e0 = identity: ✓')

# eᵢ² = −e0 for i ≥ 1
for i in range(1, 16):
    s, k = _cd_mul_basis(i, i, 4)
    assert s == -1 and k == 0, f'e{i}² should be −e0, got {s}·e{k}'
print(f'  eᵢ² = −e0 for i=1..15: ✓')

# Verify e1² = −e0 explicitly (quaternion compatibility)
s, k = _cd_mul_basis(1, 1, 4)
print(f'  e₁² = {s:+}·e{k} (= {s}·e{k}) — ✓' if s == -1 and k == 0 else f'  e₁² = {s:+}·e{k} — WRONG')

## D2 — P1: ZD Pair Count

In [ ]:
print('Finding all 84 directed ZD pairs...')
pairs = find_zd_pairs()
classification = classify_zd_pairs(pairs)

unordered = set()
for _, _, ij, kl in pairs:
    key = frozenset([frozenset(ij), frozenset(kl)])
    unordered.add(key)

print(f'Directed pairs found: {len(pairs)}')
print(f'Unordered classes:    {len(unordered)}')
print(f'Odd-sector:           {len(classification["odd_sector"])}')
print(f'Mixed:                {len(classification["mixed"])}')
print(f'Even-sector:          {len(classification["even_sector"])}')
print()

print(f'P1 PASS: {len(pairs) == 84 and len(unordered) == 42}')

# Verify all products are exactly zero
max_residual = 0.0
for a_vec, b_vec, ij, kl in pairs:
    prod = multiply(a_vec * math.sqrt(2), b_vec * math.sqrt(2))  # unnormalised
    r = float(np.max(np.abs(prod)))
    max_residual = max(max_residual, r)

print(f'Max product residual: {max_residual:.2e}  (machine zero = ✓)' if max_residual < 1e-12 else f'NONZERO: {max_residual}')

## D3 — P7: Addition = Subtraction (cross-term cancellation)

In [ ]:
print('Cross-term cancellation in all 84 ZD pairs:')
print('  (eᵢ + eⱼ)(eₖ + eₗ) = eᵢeₖ + eᵢeₗ + eⱼeₖ + eⱼeₗ = 0')
print('  Requires: eᵢeₖ = −(eⱼeₗ) AND eᵢeₗ = −(eⱼeₖ)')
print()

all_cancel = True
sample_shown = 0

for a_vec, b_vec, (i, j), (k, l) in pairs[:12]:  # show first 12
    # Four cross-term products
    t_ik = multiply(e_k(i), e_k(k))
    t_il = multiply(e_k(i), e_k(l))
    t_jk = multiply(e_k(j), e_k(k))
    t_jl = multiply(e_k(j), e_k(l))

    # Check pairwise cancellation: t_ik + t_jl = 0 AND t_il + t_jk = 0
    pair1_cancel = np.allclose(t_ik + t_jl, 0, atol=1e-12)
    pair2_cancel = np.allclose(t_il + t_jk, 0, atol=1e-12)

    if not (pair1_cancel and pair2_cancel):
        all_cancel = False

    if sample_shown < 4:
        s_ik = int(T_sign[i,k]) if i < 16 and k < 16 else '?'
        idx_ik = int(T_idx[i,k]) if i < 16 and k < 16 else '?'
        s_jl = int(T_sign[j,l]) if j < 16 and l < 16 else '?'
        idx_jl = int(T_idx[j,l]) if j < 16 and l < 16 else '?'

        from zero_lattice import _T_SIGN, _T_IDX
        s_ik  = int(_T_SIGN[i,k]);  idx_ik = int(_T_IDX[i,k])
        s_jl  = int(_T_SIGN[j,l]);  idx_jl = int(_T_IDX[j,l])
        print(f'  (e{i}+e{j})(e{k}+e{l}): e{i}·e{k}={s_ik:+}e{idx_ik}, e{j}·e{l}={s_jl:+}e{idx_jl} → sum={s_ik+s_jl if idx_ik==idx_jl else "diff basis"} ← addition=subtraction: {pair1_cancel}')
        sample_shown += 1

# Verify ALL pairs
all_cancel = True
for _, _, (i,j), (k,l) in pairs:
    t_ik = multiply(e_k(i), e_k(k))
    t_il = multiply(e_k(i), e_k(l))
    t_jk = multiply(e_k(j), e_k(k))
    t_jl = multiply(e_k(j), e_k(l))
    if not (np.allclose(t_ik + t_jl, 0, atol=1e-12) and np.allclose(t_il + t_jk, 0, atol=1e-12)):
        all_cancel = False

print(f'P7 PASS (all 84 pairs cancel): {all_cancel}')

## D4 — P2: Monster Gap Universality in Odd Sector

In [ ]:
print('Odd-sector pairs (both factors at all-odd basis indices):')
odd_pairs = classification['odd_sector']
print(f'  Count: {len(odd_pairs)}')
print()

monster_in_all_odd = True
for ij, kl in odd_pairs:
    all_idx = set(ij) | set(kl)
    has_gap = bool(all_idx & MONSTER_GAP)
    print(f'  (e{ij[0]}+e{ij[1]})(e{kl[0]}+e{kl[1]}): indices={sorted(all_idx)}, Monster gap element: {"YES ← GAP" if has_gap else "NO"}')
    if not has_gap:
        monster_in_all_odd = False

print()
print(f'P2 PASS (all odd-sector pairs involve Monster gap): {monster_in_all_odd}')

## D5 — P3: THE ANGLE Verification

In [ ]:
av = verify_angle()

print('THE ANGLE = π/8 = 22.5° (geometric definition of angle)')
print()
print(f'  J_red  before rotation:  {sorted(av["j_red_before"])}')
print(f'  J_blue before rotation:  {sorted(av["j_blue_before"])}')
print(f'  → Interleaved (offset 45°): {av["sectors_interleaved_before"]}')
print()
print(f'  J_red  after +π/8:  {sorted(av["j_red_after"])}')
print(f'  J_blue after −π/8:  {sorted(av["j_blue_after"])}')
print(f'  → Sectors COINCIDE: {av["sectors_coincide_after"]}')
print()
print(f'  tan(π/8) = √2−1 = {av["tan_exact"]:.15f}')
print(f'  tan(π/8) computed  = {math.tan(THE_ANGLE):.15f}')
print(f'  residual           = {av["tan_residual"]:.2e}')
print()
print(f'  Silver ratio check: √2−1 = {math.sqrt(2)-1:.15f}')
print(f'  16 × (π/8) = {16 * math.degrees(THE_ANGLE)}° = 1 full revolution: ✓')
print()
print(f'P3 PASS: {av["sectors_coincide_after"] and av["tan_residual"] < 1e-12}')

## D6 — P6: Spoke Wheel View

In [ ]:
tree = build_tree()
tv = view_tree(tree, computed_pairs=pairs)

print(tv['ascii_wheel'])
print()
print('Spoke assignments after THE ANGLE rotation:')
spokes = {}
for basis_idx, sp in tv['spoke_positions'].items():
    pos = round(sp['rotated_deg'], 1)
    if pos not in spokes:
        spokes[pos] = []
    spokes[pos].append(basis_idx)

for pos in sorted(spokes):
    elems = spokes[pos]
    gap_mark = any(e in MONSTER_GAP for e in elems)
    print(f'  {pos:6.1f}°: e{elems} {"← Monster gap" if gap_mark else ""}')

print()
n_positions = len(spokes)
all_4 = all(len(v) == 4 for v in spokes.values())
print(f'  Spoke positions:  {n_positions} (expect 4)')
print(f'  Elements/spoke:   {"4 each" if all_4 else "WRONG"}')
print(f'P6 PASS: {n_positions == 4 and all_4}')

## D7 — Telperion Tree Structure

In [ ]:
print('Telperion — Zero Lattice Tree (k=0 leaf → k=8 root):')
print()
print(f'{"k":>3}  {"Label":<30} {"dim":>6}  {"σ":>6}  ZD  Root/Leaf')
print('─' * 65)
for k in sorted(tree):
    n = tree[k]
    zd   = '[ZD]' if n.is_zd_level else '    '
    rl   = '← ROOT' if n.is_root else ('← LEAF' if n.is_leaf else '')
    print(f'{k:>3}  {n.label:<30} {n.dim:>6}  {n.sigma:>6.3f}  {zd}  {rl}')

## D8 — P4: Zeta Geometric on Critical Line

In [ ]:
import cmath

weights = _zd_weights(pairs)
unique_weights = sorted(set(weights))
weight_dist    = {int(w): weights.count(w) for w in set(weights)}

print('ZD pair weight distribution:')
print(f'  Unique prime weights: {unique_weights}')
print(f'  Distribution (prime: count): {dict(sorted(weight_dist.items()))}')
print(f'  Total pairs: {len(weights)}')
print()

print('Euler product ζ_T on the critical line (σ=½):')
print()
print(f'{"γ":>12}  {"Re(ζ_T)": >14}  {"Im(ζ_T)":>14}  {"│ζ_T│":>12}')
print('─' * 60)
zeta_euler_data = []
for g in RIEMANN_ZEROS[:12]:
    s_val = complex(SIGMA_HALF, g)
    ze    = zeta_geometric(s_val, pairs=pairs)
    zeta_euler_data.append({'gamma': g, 'zeta_euler': ze, 'abs': abs(ze)})
    print(f'{g:>12.6f}  {ze.real:>14.6f}  {ze.imag:>14.6f}  {abs(ze):>12.6f}')

print()
all_finite = all(math.isfinite(d['abs']) for d in zeta_euler_data)
all_nonzero = all(d['abs'] > 0.001 for d in zeta_euler_data)
print(f'P4 PASS (all finite and non-zero): {all_finite and all_nonzero}')

## D9 — Zeta Dirichlet Form

In [ ]:
print('Dirichlet series ζ_T^D(s) = Σ w^{-s} (84 terms, counting form):')
print()
print(f'{"σ":>6}  {"γ":>8}  {"│ζ_T^D│":>14}  {"│ζ_T Euler│":>14}')
print('─' * 55)

# Sample at σ = 1, ½, ¼ and some Riemann zeros
test_points = [
    (1.0,  0.0,    'σ=1 (ℂ, convergent)'),
    (0.75, 0.0,    'σ=¾ (ℂ, convergent)'),
    (0.5,  14.135, 'σ=½, γ₁ (Riemann zero 1)'),
    (0.5,  21.022, 'σ=½, γ₂ (Riemann zero 2)'),
    (0.25, 0.0,    'σ=¼ (𝕆, convergent)'),
]

for sigma, gamma, label in test_points:
    s_val = complex(sigma, gamma)
    zd_val = zeta_dirichlet(s_val, pairs=pairs)
    ze_val = zeta_geometric(s_val, pairs=pairs)
    print(f'  {sigma:>5.2f}  {gamma:>8.3f}  │ζ^D│={abs(zd_val):>10.4f}  │ζ_E│={abs(ze_val):>10.4f}  ← {label}')

## D10 — P5: Scale Switching

In [ ]:
print('Scale switching — no renormalization:')
print('  If sum diverges at σ: switch to adjacent CD level.')
print()

test_sigmas = [1.0, 0.75, 0.5, 0.25, 0.0, -0.25]
for sigma in test_sigmas:
    sw_up   = switch_scale(sigma, 'up')
    sw_down = switch_scale(sigma, 'down')
    print(f'  σ={sigma:>5.2f} (k={sw_up["k_in"]}, {sw_up["level_in"]})')
    print(f'    UP   → σ={sw_up["sigma_out"]:.2f}  k={sw_up["k_out"]}  ({sw_up["level_out"]})')
    print(f'    DOWN → σ={sw_down["sigma_out"]:.2f}  k={sw_down["k_out"]}  ({sw_down["level_out"]})')
    print()

# Demonstrate divergence at σ=0 in Dirichlet form and switch UP
print('Dirichlet series near σ=0:')
for sigma in [0.1, 0.05, 0.01]:
    s_val = complex(sigma, 0)
    zd = zeta_dirichlet(s_val, pairs=pairs)
    print(f'  σ={sigma:.3f}: │ζ^D│ = {abs(zd):.4f}')

print()
print('At σ=0, switch UP to σ=0.25 (𝕆 level):')
sw = switch_scale(0.0, 'up')
print(f'  {sw}')
print(f'P5 PASS: switch applied without renormalization ✓')

## D11 — Monster Gap in Map

In [ ]:
gap = monster_gap_in_map()

print(f'Monster gap elements: {gap["gap_elements"]}')
print(f'Shell spread: {gap["shell_spread"]} different shells')
print()
for k, cell in gap['cells'].items():
    print(f'  e{k}: Shell {cell["shell"]}, σ={cell["sigma"]}, {cell["angle_deg"]}°, {cell["j_type"]}')

print()
print(gap['interpretation'])
print()
print(f'Moonshine primes: {gap["moonshine_primes"]}')

## D12 — P8: Universal Translator Structure

In [ ]:
trans = universal_translator_structure()

print('Universal Translator — Zero Lattice structure:')
print()
print(f'  Claim:           {trans["claim"]}')
print(f'  Mechanism:       {trans["mechanism"]}')
print(f'  Public key role: {trans["public_key_role"]}')
print(f'  Private key role:{trans["private_key_role"]}')
print(f'  Path:            {trans["path"]}')
print(f'  Exactness:       {trans["exactness"]}')
print(f'  Reference:       {trans["reference"]}')

print()
print('Laurelin interface (two trees, one world):')
li = laurelin_interface()
print(f'  Shared leaves:   {li["shared_leaves"]}')
print(f'  Shared root:     {li["shared_root"]}')
print(f'  Cooperation at:  {li["cooperation_at"]}')
print(f'  Completion:      {li["completion"]}')

## D13 — Critical Line Scan

In [ ]:
print('Sampling |ζ_T(½+iγ)| across γ ∈ [0, 80]...')
sample_data = critical_line_samples(0.0, 80.0, 2000)

print(f'  σ = {sample_data["sigma"]}')
print(f'  γ range: {sample_data["gamma_range"]}')
print(f'  Points sampled: {sample_data["n_points"]}')
print(f'  Local minima found: {len(sample_data["local_minima"])}')
print()

print('Riemann zero comparison:')
print(f'{"γ (Riemann)":>15}  {"│ζ_T│ at γ":>14}')
print('─' * 35)
for entry in sample_data['riemann_zeros']:
    val = entry['zeta_T_value']
    print(f'  {entry["gamma"]:>12.6f}  {val:>12.6f}' if val else f'  {entry["gamma"]:>12.6f}  N/A')

if sample_data['local_minima']:
    print()
    print('Local minima (approximate ζ_T zeros):')
    for lm in sample_data['local_minima'][:10]:
        print(f'  γ ≈ {lm["gamma"]:.4f}, │ζ_T│ ≈ {lm["value"]:.6f}')

## D14 — Full Engine Run (run_all)

In [ ]:
results = run_all()

## D15 — ZD Path Coordinates (sample traversals)

In [ ]:
print('Sample ZD path coordinates in the Sedenion Point Mapping (odd sector):')
print()

sample_constellations = [
    (1, 11,  5, 15),  # first odd-sector
    (1,  9,  3, 11),  # second
    (7, 15,  9,  1),  # third
]

for const in sample_constellations:
    path = zd_path_coordinates(const)
    a, b, c, d = const
    print(f'  (e{a}+e{b})(e{c}+e{d}) = 0:')
    for node in path:
        if 'label' in node:
            sigma  = node.get('sigma', '?')
            angle  = node.get('angle_deg', '?')
            shell  = node.get('shell', '?')
            jtype  = node.get('j_type', '?')
            gap    = ' ← GAP' if node.get('is_monster') else ''
            print(f'    {node["label"]:>4}: Shell {shell}, σ={sigma:.2f}, {angle:>5}°, {jtype}{gap}')
        else:
            print(f'    {node["point"]}')
    print()